# Ion-expert single-fragment diagnostics

This companion notebook keeps only the isolated fragment-expert checks for the ion expert trained at `checkpoints/ion_expert_full`: one-body energies, molecular multipoles, polarizabilities, and true one-body forces where the labels exist.

It intentionally leaves cluster interaction components and competing-fragmentation plots to `ion_expert_eda_plotting.ipynb`.


In [ ]:
%matplotlib inline

import os
import re
import sys
from pathlib import Path

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/rsfff_matplotlib")

import matplotlib.pyplot as plt
import numpy as np
import torch

# Resolve the repository root no matter where Jupyter starts the kernel.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
for p in (ROOT / "src", ROOT / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from rsfff.ff.molecular_multipoles import fragment_multipoles, reference_multipoles
from rsfff.ff.multipole import spherical_to_cartesian_quadrupole
from rsfff.ff.units import BOHR_ANG, KJMOL_PER_HARTREE
from rsfff.mlip.heads import env_parameters
from rsfff.mlip.reference_states import AtomicStateReference
from rsfff.train.build_expert import build_expert_model
from rsfff.train.config import load_config
from rsfff.train.data import (
    fragment_view,
    load_cluster_datasets as load_cluster_dataset_expanded,
    load_datasets,
    load_extxyz,
    load_reference_energies,
    split_indices_grouped,
)
from rsfff.train.loss import compute_forces
from rsfff.train.train_eem import resolve_device

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 220,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e6e8ef",
    "grid.linewidth": 0.7,
    "axes.labelcolor": "#20242c",
    "axes.edgecolor": "#aeb4c0",
    "xtick.color": "#3c4450",
    "ytick.color": "#3c4450",
    "font.size": 9,
})


In [ ]:
# ---- Regeneration knobs ---------------------------------------------------------------
CHECKPOINT = "checkpoints/ion_expert_full/best.pt"
CONFIG = "configs/ion_expert.yaml"       # fallback only; checkpoint config is preferred
DEVICE = "cpu"                           # CPU is stable for float64 plots; use "auto" if desired
EVAL_SPLIT = "all"                       # used only for optional cluster fragment views
MAX_FRAMES_PER_SOURCE = None              # e.g. 300 for quick iteration
BATCH_SIZE = 64
FORCE_FRAMES_PER_SOURCE = 80              # forces need autograd; keep this modest for interactivity
INCLUDE_CLUSTER_FRAGMENT_VIEWS = True      # adds the isolated fragment labels harvested from EDA clusters
FIGDIR = Path("notebooks/figures")
FIGDIR.mkdir(parents=True, exist_ok=True)
FIGPREFIX = Path(CHECKPOINT).parent.name

ELEMENT = {1: "H", 8: "O"}


In [ ]:
def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def as_paths(value):
    if value is None:
        return []
    return value if isinstance(value, list) else [value]


def safe_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")


def dataset_tag(path):
    stem = Path(path).stem
    for suffix in ("_wb97mv_tzvpd_pol", "_wb97mv_tzvpd", "_wb97mv", "_tzvpd"):
        if stem.endswith(suffix):
            stem = stem[: -len(suffix)]
    return stem


def stage_of(cfg):
    for stage in cfg.stages:
        if cfg.run_name.endswith(f"_{stage.name}"):
            return stage.name
    return None


def load_expert_checkpoint(checkpoint=CHECKPOINT, device=DEVICE, config_fallback=CONFIG):
    """Rebuild the model from the checkpoint's embedded config, falling back to YAML only for old checkpoints."""
    state = torch_load(checkpoint)
    cfg = state.get("config") or load_config(config_fallback)
    torch.set_default_dtype(torch.float64 if cfg.dtype == "float64" else torch.float32)

    neighbor_types = state.get("neighbor_types")
    if neighbor_types is None:
        probe = load_datasets(cfg.data.path, dtype=torch.get_default_dtype())
        neighbor_types = probe.unique_atomic_numbers
    neighbor_types = tuple(int(z) for z in neighbor_types)

    reference_energies = load_reference_energies(cfg.data.reference_energies, neighbor_types).to(torch.get_default_dtype())
    atomic_states = None
    if cfg.data.atomic_reference_states:
        atomic_states = AtomicStateReference.from_json(
            cfg.data.atomic_reference_states,
            neighbor_types,
            dtype=torch.get_default_dtype(),
        )
    model = build_expert_model(cfg, neighbor_types, reference_energies, atomic_states)
    model.load_state_dict(state["model_state"], strict=True)
    model.eval()
    resolved = torch.device("cpu" if device == "cpu" else resolve_device(device, cfg.dtype))
    model.to(resolved)
    return model, cfg, state, stage_of(cfg), resolved


def load_cluster_datasets_by_file(cfg):
    dtype = torch.float64 if cfg.dtype == "float64" else torch.float32
    out = {}
    for p in as_paths(cfg.data.path):
        out[dataset_tag(p)] = load_cluster_dataset_expanded(
            [p], dtype=dtype, fragmentations=cfg.data.fragmentations
        )
    return out


def frame_indices(dataset, cfg):
    n = len(dataset)
    if EVAL_SPLIT == "validation":
        if getattr(dataset, "_group_id", None) is not None:
            _, val = split_indices_grouped(dataset._group_id, cfg.data.holdout_fraction, cfg.data.seed)
            idx = np.sort(val.cpu().numpy())
        else:
            rng = np.random.default_rng(cfg.data.seed)
            perm = rng.permutation(n)
            n_val = max(1, int(round(n * cfg.data.holdout_fraction)))
            idx = np.sort(perm[:n_val])
    elif EVAL_SPLIT == "all":
        idx = np.arange(n)
    else:
        raise ValueError("EVAL_SPLIT must be 'all' or 'validation'")
    if MAX_FRAMES_PER_SOURCE is not None:
        idx = idx[:MAX_FRAMES_PER_SOURCE]
    return idx.tolist()


def to_np(x):
    return x.detach().cpu().numpy()


def fragment_to_system(batch):
    if batch.fragment_to_batch is not None:
        return batch.fragment_to_batch
    f2b = batch.batch_idx.new_zeros(batch.n_fragments)
    return f2b.scatter_(0, batch.fragment_idx, batch.batch_idx)


def pool_fragments_to_frames(values, batch):
    f2b = fragment_to_system(batch)
    return values.new_zeros(batch.n_systems).index_add_(0, f2b, values)


def fragments_per_frame(batch):
    f2b = fragment_to_system(batch)
    return torch.bincount(f2b, minlength=batch.n_systems).to(batch.positions.dtype)


def pooled_applicability(out, batch):
    scores = getattr(out, "applicability", None)
    if scores is None:
        return None
    f2b = fragment_to_system(batch)
    total = scores.new_zeros(batch.n_systems).index_add_(0, f2b, scores)
    count = scores.new_zeros(batch.n_systems).index_add_(0, f2b, torch.ones_like(scores))
    return total / count.clamp(min=1.0)


def metrics(ref, pred):
    ref = np.asarray(ref)
    pred = np.asarray(pred)
    err = pred - ref
    if len(ref) > 1 and np.std(ref) > 0 and np.std(pred) > 0:
        r2 = float(np.corrcoef(ref, pred)[0, 1] ** 2)
    else:
        r2 = np.nan
    return float(np.abs(err).mean()), float(np.sqrt(np.mean(err ** 2))), r2


PALETTE = {
    "w1_h3o+": "#276ef1",
    "w2_h3o+": "#0f8b8d",
    "w1_oh-": "#d64545",
    "w2_oh-": "#db6d00",
    "w2": "#6f58c9",
    "w3": "#1b998b",
    "w4": "#b05c00",
    "w5": "#7a4cc2",
    "h2o": "#276ef1",
    "h2o_opt": "#1b998b",
    "h3o+": "#0f8b8d",
    "h3o+_opt": "#47a9cf",
    "oh-": "#d64545",
    "oh-_opt": "#db6d00",
}


In [ ]:
def load_monomer_datasets_by_file(cfg):
    dtype = torch.float64 if cfg.dtype == "float64" else torch.float32
    return {dataset_tag(p): load_extxyz(p, dtype=dtype) for p in as_paths(cfg.data.monomer_path)}


def load_fragment_view_datasets(cfg):
    clusters = load_cluster_datasets_by_file(cfg)
    views = {}
    for tag, ds in clusters.items():
        if not ds.has_fragments or getattr(ds, "_fragment_energy", None) is None:
            continue
        idx = frame_indices(ds, cfg)
        views[f"{tag}_fragments"] = fragment_view(ds, idx)
    return views


model, cfg, state, stage_name, device = load_expert_checkpoint()
monomer_datasets = load_monomer_datasets_by_file(cfg)
fragment_view_datasets = load_fragment_view_datasets(cfg) if INCLUDE_CLUSTER_FRAGMENT_VIEWS else {}
fragment_datasets = {**monomer_datasets, **fragment_view_datasets}

print(f"checkpoint: {CHECKPOINT}")
print(f"stage: {stage_name or 'single'}    run_name: {cfg.run_name}")
print(f"epoch: {state.get('epoch', 'unknown')}    val_loss: {state.get('val_loss', float('nan')):.6g}")
print(f"device: {device}    dtype: {cfg.dtype}")
print(f"experts: {sorted(model.experts.experts)}")
print("monomer datasets:", {tag: len(ds) for tag, ds in monomer_datasets.items()})
print("fragment-view datasets:", {tag: len(ds) for tag, ds in fragment_view_datasets.items()})


## One-Body Energies

One-body energy labels come from both the dedicated monomer/ion files and, optionally, the isolated fragment views harvested from the cluster EDA files. The parity panel is source-centered for readability; the residual panel preserves absolute offsets.


In [ ]:
def collect_fragment_energy_predictions(model, datasets, device, batch_size=BATCH_SIZE):
    rows = {"ref": [], "pred": [], "source": [], "n_atoms": [], "charge": []}
    for source, ds in datasets.items():
        indices = list(range(len(ds)))
        if MAX_FRAMES_PER_SOURCE is not None:
            indices = indices[:MAX_FRAMES_PER_SOURCE]
        for start in range(0, len(indices), batch_size):
            batch = ds.flat_batch(indices[start:start + batch_size]).to(device)
            with torch.no_grad():
                out = model(batch, with_induction=False)
            pred = pool_fragments_to_frames(out.fragment_energy, batch)
            if batch.fragment_energy is not None:
                ref = pool_fragments_to_frames(batch.fragment_energy, batch)
            else:
                ref = batch.energy
            counts = torch.bincount(batch.batch_idx, minlength=batch.n_systems)
            charge = batch.total_charge if batch.total_charge is not None else pred.new_full((batch.n_systems,), np.nan)
            rows["pred"].append(to_np(pred) * KJMOL_PER_HARTREE)
            rows["ref"].append(to_np(ref) * KJMOL_PER_HARTREE)
            rows["n_atoms"].append(to_np(counts))
            rows["charge"].append(to_np(charge))
            rows["source"].extend([source] * len(pred))
    return {
        "pred": np.concatenate(rows["pred"]),
        "ref": np.concatenate(rows["ref"]),
        "source": np.asarray(rows["source"]),
        "n_atoms": np.concatenate(rows["n_atoms"]),
        "charge": np.concatenate(rows["charge"]),
    }


fragment_energy = collect_fragment_energy_predictions(model, fragment_datasets, device)
mae, rmse, r2 = metrics(fragment_energy["ref"], fragment_energy["pred"])
print(f"one-body energies: n={len(fragment_energy['ref'])}  MAE={mae:.5g} kJ/mol  RMSE={rmse:.5g}  R2={r2:.5f}")
print(f"{'source':<22}{'n':>7}{'MAE':>12}{'bias':>12}")
for source in sorted(set(fragment_energy["source"])):
    m = fragment_energy["source"] == source
    err = fragment_energy["pred"][m] - fragment_energy["ref"][m]
    print(f"{source:<22}{m.sum():>7}{np.abs(err).mean():>12.4f}{err.mean():>12.4f}")


In [ ]:
def plot_fragment_energy(fragment_energy, path=None):
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))

    ax = axes[0]
    ref_c = np.zeros_like(fragment_energy["ref"])
    pred_c = np.zeros_like(fragment_energy["pred"])
    for source in sorted(set(fragment_energy["source"])):
        m = fragment_energy["source"] == source
        offset = fragment_energy["ref"][m].mean()
        ref_c[m] = fragment_energy["ref"][m] - offset
        pred_c[m] = fragment_energy["pred"][m] - offset
        ax.scatter(ref_c[m], pred_c[m], s=8, alpha=0.5, lw=0,
                   color=PALETTE.get(source.replace("_fragments", ""), "#555"), label=source)
    lo = min(ref_c.min(), pred_c.min())
    hi = max(ref_c.max(), pred_c.max())
    pad = 0.05 * (hi - lo if hi > lo else 1.0)
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color="#242934", lw=1, ls=(0, (4, 3)))
    mae, rmse, r2 = metrics(fragment_energy["ref"], fragment_energy["pred"])
    ax.text(0.04, 0.96, f"absolute MAE {mae:.3g}\nRMSE {rmse:.3g}\nR2 {r2:.4f}",
            transform=ax.transAxes, ha="left", va="top", fontsize=8,
            bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "#d6dae3", "alpha": 0.85})
    ax.set_xlabel("reference one-body energy, source-centered (kJ/mol)")
    ax.set_ylabel("predicted one-body energy, same offset (kJ/mol)")
    ax.set_title("(a) geometry-dependent one-body energy")
    ax.legend(frameon=False, fontsize=6, ncol=2)

    ax = axes[1]
    for source in sorted(set(fragment_energy["source"])):
        m = fragment_energy["source"] == source
        err = fragment_energy["pred"][m] - fragment_energy["ref"][m]
        ax.hist(err, bins=50, histtype="step", lw=1.2,
                color=PALETTE.get(source.replace("_fragments", ""), "#555"), label=source)
    ax.axvline(0.0, color="#242934", lw=1, ls=(0, (4, 3)))
    ax.set_xlabel("predicted - reference one-body energy (kJ/mol)")
    ax.set_ylabel("frames")
    ax.set_title("(b) one-body residuals")
    ax.legend(frameon=False, fontsize=6, ncol=2)

    fig.suptitle("Fragment expert one-body energy", y=1.03, fontsize=13)
    fig.tight_layout()
    if path:
        fig.savefig(path, bbox_inches="tight")
    return fig


fig = plot_fragment_energy(fragment_energy, path=FIGDIR / f"{FIGPREFIX}_fragment_onebody_energy.png")
plt.show()


## Multipoles and Polarizabilities

The frozen response solve provides the permanent multipoles and analytic polarizability. Only datasets carrying the corresponding labels contribute to each panel.


In [ ]:
def collect_response_properties(model, datasets, device, batch_size=BATCH_SIZE):
    chunks = {
        "polarizability": {"ref": [], "pred": [], "component": [], "source": []},
        "dipole": {"ref": [], "pred": [], "component": [], "source": []},
        "quadrupole": {"ref": [], "pred": [], "component": [], "source": []},
    }
    tri = np.triu_indices(3)
    alpha_labels = np.array(["xx", "xy", "xz", "yy", "yz", "zz"])
    dip_labels = np.array(["x", "y", "z"])
    quad_labels = alpha_labels

    for source, ds in datasets.items():
        indices = list(range(len(ds)))
        if MAX_FRAMES_PER_SOURCE is not None:
            indices = indices[:MAX_FRAMES_PER_SOURCE]
        for start in range(0, len(indices), batch_size):
            batch = ds.flat_batch(indices[start:start + batch_size]).to(device)
            with torch.no_grad():
                out = model(batch, with_polarizability=True, with_induction=False)
            if out.polarizability is not None and batch.polarizability is not None:
                pred_a = to_np(out.polarizability)[:, tri[0], tri[1]] / (BOHR_ANG ** 2)
                ref_a = to_np(batch.polarizability)[:, tri[0], tri[1]] / (BOHR_ANG ** 2)
                chunks["polarizability"]["pred"].append(pred_a.ravel())
                chunks["polarizability"]["ref"].append(ref_a.ravel())
                chunks["polarizability"]["component"].extend(np.tile(alpha_labels, pred_a.shape[0]))
                chunks["polarizability"]["source"].extend([source] * pred_a.size)

            if batch.fragment_dipole is not None and batch.fragment_second_moment is not None:
                quad_c = None if out.quad_s is None else spherical_to_cartesian_quadrupole(out.quad_s)
                frag_q = batch.fragment_charge if batch.fragment_charge is not None else batch.positions.new_zeros(batch.n_fragments)
                pred_d, pred_q = fragment_multipoles(
                    out.charges, batch.positions, batch.atomic_numbers, batch.fragment_idx,
                    batch.n_fragments, frag_q, out.mu, quad_c,
                )
                ref_d, ref_q = reference_multipoles(
                    batch.fragment_dipole, batch.fragment_second_moment, batch.positions,
                    batch.atomic_numbers, batch.fragment_idx, batch.n_fragments, frag_q,
                )
                pred_d, ref_d = to_np(pred_d), to_np(ref_d)
                pred_q, ref_q = to_np(pred_q)[:, tri[0], tri[1]], to_np(ref_q)[:, tri[0], tri[1]]
                chunks["dipole"]["pred"].append(pred_d.ravel())
                chunks["dipole"]["ref"].append(ref_d.ravel())
                chunks["dipole"]["component"].extend(np.tile(dip_labels, pred_d.shape[0]))
                chunks["dipole"]["source"].extend([source] * pred_d.size)
                chunks["quadrupole"]["pred"].append(pred_q.ravel())
                chunks["quadrupole"]["ref"].append(ref_q.ravel())
                chunks["quadrupole"]["component"].extend(np.tile(quad_labels, pred_q.shape[0]))
                chunks["quadrupole"]["source"].extend([source] * pred_q.size)

    packed = {}
    for key, data in chunks.items():
        if data["pred"]:
            packed[key] = {
                "pred": np.concatenate(data["pred"]),
                "ref": np.concatenate(data["ref"]),
                "component": np.asarray(data["component"]),
                "source": np.asarray(data["source"]),
            }
    return packed


properties = collect_response_properties(model, fragment_datasets, device)
for key, d in properties.items():
    mae, rmse, r2 = metrics(d["ref"], d["pred"])
    print(f"{key:<16} n={len(d['ref']):<7} MAE={mae:.5g} RMSE={rmse:.5g} R2={r2:.5f}")


In [ ]:
def plot_property_correlations(properties, path=None):
    labels = {
        "polarizability": ("polarizability tensor", "bohr^3"),
        "dipole": ("fragment dipole", "e bohr"),
        "quadrupole": ("Buckingham quadrupole", "e bohr^2"),
    }
    keys = [k for k in labels if k in properties]
    fig, axes = plt.subplots(1, len(keys), figsize=(4.4 * len(keys), 3.9), squeeze=False)
    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    for ax, key in zip(axes.ravel(), keys):
        d = properties[key]
        ref, pred, comp = d["ref"], d["pred"], d["component"]
        lo = min(ref.min(), pred.min())
        hi = max(ref.max(), pred.max())
        pad = 0.05 * (hi - lo if hi > lo else 1.0)
        ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color="#242934", lw=1, ls=(0, (4, 3)))
        for i, c in enumerate(sorted(set(comp))):
            m = comp == c
            ax.scatter(ref[m], pred[m], s=7, alpha=0.48, lw=0, color=colors[i % len(colors)], label=c)
        mae, rmse, r2 = metrics(ref, pred)
        ax.text(0.04, 0.96, f"MAE {mae:.3g}\nRMSE {rmse:.3g}\nR2 {r2:.4f}",
                transform=ax.transAxes, ha="left", va="top", fontsize=8,
                bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "#d6dae3", "alpha": 0.85})
        ax.set_title(labels[key][0])
        ax.set_xlabel(f"reference ({labels[key][1]})")
        ax.set_ylabel(f"predicted ({labels[key][1]})")
        ax.set_xlim(lo - pad, hi + pad)
        ax.set_ylim(lo - pad, hi + pad)
        ax.legend(frameon=False, fontsize=7, ncol=2)
    fig.suptitle("Single-fragment response-property correlations", y=1.03, fontsize=13)
    fig.tight_layout()
    if path:
        fig.savefig(path, bbox_inches="tight")
    return fig


if properties:
    fig = plot_property_correlations(properties, path=FIGDIR / f"{FIGPREFIX}_fragment_response_properties.png")
    plt.show()


## One-Body Forces

These are evaluated on the dedicated single-fragment files that carry true nuclear force labels. Cluster fragment views are excluded here because cluster forces are gradients of the whole cluster, not isolated fragment forces.


In [ ]:
def collect_fragment_force_predictions(model, datasets, device, n_frames=FORCE_FRAMES_PER_SOURCE, batch_size=8):
    rows = {"ref": [], "pred": [], "source": []}
    for source, ds in datasets.items():
        if not ds.has_forces:
            print(f"{source}: no force labels; skipping")
            continue
        indices = list(range(min(n_frames, len(ds))))
        for start in range(0, len(indices), batch_size):
            batch = ds.flat_batch(indices[start:start + batch_size]).to(device)
            batch.positions.requires_grad_(True)
            out = model(batch, with_induction=False)
            forces = compute_forces(out.energy, batch.positions, create_graph=False)
            ref = to_np(batch.forces) * KJMOL_PER_HARTREE
            pred = to_np(forces) * KJMOL_PER_HARTREE
            rows["ref"].append(ref.reshape(-1))
            rows["pred"].append(pred.reshape(-1))
            rows["source"].extend([source] * ref.size)
    if not rows["pred"]:
        return {}
    return {"ref": np.concatenate(rows["ref"]), "pred": np.concatenate(rows["pred"]), "source": np.asarray(rows["source"])}


force_sources = monomer_datasets
forces = collect_fragment_force_predictions(model, force_sources, device)
if forces:
    mae, rmse, r2 = metrics(forces["ref"], forces["pred"])
    print(f"force components: n={len(forces['ref'])}  MAE={mae:.5g} kJ/mol/A  RMSE={rmse:.5g}  R2={r2:.5f}")


In [ ]:
def plot_force_correlation(forces, path=None):
    fig, ax = plt.subplots(figsize=(5.4, 5.0))
    ref, pred, sources = forces["ref"], forces["pred"], forces["source"]
    lo = min(ref.min(), pred.min())
    hi = max(ref.max(), pred.max())
    pad = 0.05 * (hi - lo if hi > lo else 1.0)
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color="#242934", lw=1, ls=(0, (4, 3)))
    for source in sorted(set(sources)):
        m = sources == source
        ax.scatter(ref[m], pred[m], s=5, alpha=0.32, lw=0, color=PALETTE.get(source, "#555"), label=source)
    mae, rmse, r2 = metrics(ref, pred)
    ax.text(0.04, 0.96, f"MAE {mae:.3g}\nRMSE {rmse:.3g}\nR2 {r2:.4f}",
            transform=ax.transAxes, ha="left", va="top", fontsize=8,
            bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "#d6dae3", "alpha": 0.85})
    ax.set_xlabel("reference force component (kJ/mol/A)")
    ax.set_ylabel("predicted force component (kJ/mol/A)")
    ax.set_title("Single-fragment force correlation")
    ax.set_xlim(lo - pad, hi + pad)
    ax.set_ylim(lo - pad, hi + pad)
    ax.legend(frameon=False, fontsize=7, markerscale=2)
    fig.tight_layout()
    if path:
        fig.savefig(path, bbox_inches="tight")
    return fig


if forces:
    fig = plot_force_correlation(forces, path=FIGDIR / f"{FIGPREFIX}_fragment_force_correlation.png")
    plt.show()


## Notes

- The model is evaluated with `with_induction=False` for these single-fragment diagnostics. A lone fragment can otherwise relax against its own field, while the isolated labels are frozen-fragment quantities.
- Energy parity is source-centered in panel (a) because H2O, H3O+, and HO have very different absolute electronic energies. Panel (b) keeps the absolute residual, so constant offsets remain visible.
- `INCLUDE_CLUSTER_FRAGMENT_VIEWS=True` adds the per-fragment labels harvested from EDA cluster files. Turn it off to look only at the dedicated monomer/ion anchor files.
